<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/01-python-foundations/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1: Solutions

Try the exercises before reading these. For each problem there's a solution, the check that confirms it works, and some notes on why it's written that way and where the idea shows up in real ML work.

Your solution doesn't have to match mine. If it passes the checks and you can read it easily a week from now, it's a good solution.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below. You don't need to read it,
# but feel free to look at the test cases if you're stuck.

import copy
import math


class Raises:
    """Marks a test case where the function is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, float) or isinstance(b, float):
        if not isinstance(a, (int, float)) or not isinstance(b, (int, float)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=60):
    """repr() that stays readable for long lists."""
    text = repr(x)
    if len(text) <= limit:
        return text
    if isinstance(x, tuple):
        return "(" + ", ".join(_short(item, limit) for item in x) + ")"
    if isinstance(x, list):
        head = ", ".join(repr(item) for item in x[:3])
        return f"[{head}, ... {len(x)} items]"
    return text[:limit - 3] + "..."


def _call_str(func, args):
    return f"{func.__name__}({', '.join(_short(a) for a in args)})"


def _run(func, cases):
    for args, expected, hint in cases:
        call = _call_str(func, args)
        before = copy.deepcopy(args)
        try:
            result = func(*args)
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {call}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {expected!r}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {call}")
            print(f"      expected {expected!r}, but it returned {result!r}")
            if hint:
                print(f"      hint: {hint}")
            return
        if args != before:
            print(f"FAIL  {call}")
            print("      your function modified its input. It should leave it unchanged.")
            return
        if not _same(result, expected):
            print(f"FAIL  {call}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {func.__name__}")


def check_accuracy(f):
    _run(f, [
        (([1, 0, 1, 1], [1, 0, 0, 1]), 0.75, None),
        (([1, 1, 1], [1, 1, 1]), 1.0, None),
        (([0, 0], [1, 1]), 0.0, None),
        ((["cat", "dog", "cat"], ["cat", "cat", "cat"]), 2 / 3,
         "labels don't have to be numbers"),
    ])


def check_clip(f):
    _run(f, [
        ((5, 0, 10), 5, None),
        ((-3, 0, 10), 0, None),
        ((15, 0, 10), 10, None),
        ((0.5, 0.0, 1.0), 0.5, None),
        ((1e-12, 1e-7, 1 - 1e-7), 1e-7, None),
    ])


def check_is_valid_probability(f):
    _run(f, [
        ((0.5,), True, None),
        ((0,), True, "0 is a valid probability"),
        ((1,), True, "1 is a valid probability"),
        ((1.0001,), False, None),
        ((-0.1,), False, None),
        ((float("nan"),), False, "NaN is not a valid probability. Check how NaN behaves in comparisons."),
    ])


def check_count_labels(f):
    _run(f, [
        (([1, 0, 1, 1],), {1: 3, 0: 1}, None),
        (([],), {}, "an empty list should give an empty dict"),
        ((["cat", "dog", "cat"],), {"cat": 2, "dog": 1}, None),
    ])


def check_normalize(f):
    _run(f, [
        (([0, 5, 10],), [0.0, 0.5, 1.0], None),
        (([2, 4],), [0.0, 1.0], None),
        (([-1, 0, 1],), [0.0, 0.5, 1.0], None),
        (([3, 3, 3],), [0.0, 0.0, 0.0], "what happens to max - min when all values are equal?"),
        (([10],), [0.0], None),
    ])


def check_mean_and_std(f):
    _run(f, [
        (([1, 2, 3, 4, 5],), (3.0, math.sqrt(2)), "use the population std: divide by n, not n - 1"),
        (([5, 5, 5],), (5.0, 0.0), None),
        (([2, 4, 4, 4, 5, 5, 7, 9],), (5.0, 2.0), None),
        (([-1, 1],), (0.0, 1.0), None),
    ])


def check_one_hot(f):
    _run(f, [
        (("dog", ["cat", "dog", "bird"]), [0, 1, 0], None),
        (("bird", ["cat", "dog", "bird"]), [0, 0, 1], None),
        ((2, [0, 1, 2, 3]), [0, 0, 1, 0], None),
        (("fish", ["cat", "dog", "bird"]), Raises(ValueError), "raise a ValueError for a label that isn't in classes"),
    ])


def check_train_test_split(f):
    _run(f, [
        ((list(range(10)), 0.2), (list(range(8)), [8, 9]), None),
        ((["a", "b", "c", "d"], 0.5), (["a", "b"], ["c", "d"]), None),
        ((list(range(10)), 0.0), (list(range(10)), []), "with test_ratio=0 the test set is empty. Careful: data[-0:] is the whole list"),
        ((list(range(5)), 1.0), ([], list(range(5))), None),
        ((list(range(100)), 0.29), (list(range(71)), list(range(71, 100))),
         "100 * 0.29 is 28.999999999999996 in floating point. Use round(), not int()"),
    ])


def check_confusion_counts(f):
    _run(f, [
        (([1, 0, 1, 1, 0, 0], [1, 1, 0, 1, 0, 0]), {"tp": 2, "fp": 1, "tn": 2, "fn": 1}, None),
        (([1, 1], [1, 1]), {"tp": 2, "fp": 0, "tn": 0, "fn": 0}, "include all four keys, even when a count is 0"),
        (([0, 0, 0], [1, 1, 1]), {"tp": 0, "fp": 3, "tn": 0, "fn": 0}, None),
    ])


def check_precision_recall_f1(f):
    _run(f, [
        (([1, 0, 1, 1, 0, 0], [1, 1, 0, 1, 0, 0]), (2 / 3, 2 / 3, 2 / 3), None),
        (([1, 1, 1, 0], [1, 0, 0, 0]), (1.0, 1 / 3, 0.5), None),
        (([0, 0, 0], [0, 0, 0]), (0.0, 0.0, 0.0), "no positive predictions and no positive labels: return 0.0 instead of dividing by zero"),
        (([1, 0], [0, 1]), (0.0, 0.0, 0.0), "precision and recall are both 0 here, so the F1 formula divides by zero"),
    ])


def check_moving_average(f):
    _run(f, [
        (([1, 2, 3, 4, 5], 2), [1.5, 2.5, 3.5, 4.5], None),
        (([1, 2, 3, 4, 5], 5), [3.0], None),
        (([4, 8, 6], 1), [4.0, 8.0, 6.0], None),
        (([1, 2, 3], 0), Raises(ValueError), "a window of 0 makes no sense"),
        (([1, 2, 3], 4), Raises(ValueError), "the window can't be bigger than the list"),
    ])


def check_batch(f):
    _run(f, [
        ((list(range(7)), 3), [[0, 1, 2], [3, 4, 5], [6]], None),
        ((list(range(7)), 3, True), [[0, 1, 2], [3, 4, 5]], None),
        ((list(range(6)), 2), [[0, 1], [2, 3], [4, 5]], None),
        (([], 4), [], None),
        (([1, 2], 5), [[1, 2]], None),
        (([1, 2], 5, True), [], "the only batch is incomplete, so drop_last removes it"),
        ((list(range(3)), 0), Raises(ValueError), "batch_size must be at least 1"),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `accuracy`

In [2]:
def accuracy(y_true, y_pred):
    correct = sum(t == p for t, p in zip(y_true, y_pred))
    return correct / len(y_true)

In [3]:
check_accuracy(accuracy)

PASS  all 4 tests passed for accuracy


`t == p` gives `True` or `False`, and `sum` treats those as 1 and 0, so summing the comparisons counts the matches. A plain loop with a counter works just as well.

In real code you'd also check that the lengths match, like we did in section 8 of the lesson. Here the exercise said we could assume they do.

### Exercise 2: `clip`

In [4]:
def clip(value, low, high):
    return max(low, min(value, high))

In [5]:
check_clip(clip)

PASS  all 5 tests passed for clip


`min(value, high)` caps the value from above, then `max(low, ...)` raises it from below. Read it from the inside out.

An `if/elif/else` version is completely fine and arguably easier to read:

```python
if value < low:
    return low
if value > high:
    return high
return value
```

NumPy has this built in as `np.clip`, and PyTorch has `torch.clamp`.

### Exercise 3: `is_valid_probability`

In [6]:
def is_valid_probability(p):
    return 0 <= p <= 1

In [7]:
check_is_valid_probability(is_valid_probability)

PASS  all 6 tests passed for is_valid_probability


The chained comparison handles NaN correctly without any extra work, because every comparison involving NaN is `False`.

But watch what happens if you write the same logic the other way around:

```python
def is_valid_probability(p):
    return not (p < 0 or p > 1)
```

For NaN, both `p < 0` and `p > 1` are `False`, so this returns `not False`, which is `True`. Mathematically the two versions look equivalent. With NaN they aren't. This is the kind of thing that lets bad values slip through a validation check.

### Exercise 4: `count_labels`

In [8]:
def count_labels(labels):
    counts = {}
    for label in labels:
        counts[label] = counts.get(label, 0) + 1
    return counts

In [9]:
check_count_labels(count_labels)

PASS  all 3 tests passed for count_labels


`counts.get(label, 0)` returns the current count, or 0 if the label hasn't been seen yet. That saves you an `if label in counts` check.

This is one of the first things to run on any classification dataset. If one class makes up 95% of your data, a model that always predicts that class gets 95% accuracy while being useless. You need to know that before you start training.

The standard library has `collections.Counter`, which does exactly this. We'll use it in Module 2.

## Medium

### Exercise 5: `normalize`

In [10]:
def normalize(values):
    lo, hi = min(values), max(values)
    if hi == lo:
        return [0.0 for _ in values]
    return [(v - lo) / (hi - lo) for v in values]

In [11]:
check_normalize(normalize)

PASS  all 5 tests passed for normalize


Compute `min` and `max` once, before the comprehension. If you write `[(v - min(values)) / (max(values) - min(values)) for v in values]`, Python recomputes them for every element, which turns an O(n) function into an O(n²) one. On a list of a million values you'd notice.

Using `==` on floats is fine here, because we're checking whether two values taken from the same list are identical, not whether a computed result equals some target.

One important point for real projects: compute `min` and `max` from the training data only, then use those same numbers to scale the test data. If you scale the test set with its own min and max, information about the test set leaks into your preprocessing. You'll see this done properly in the mini project.

### Exercise 6: `mean_and_std`

In [12]:
def mean_and_std(values):
    n = len(values)
    mean = sum(values) / n
    variance = sum((v - mean) ** 2 for v in values) / n
    return mean, variance ** 0.5

In [13]:
check_mean_and_std(mean_and_std)

PASS  all 4 tests passed for mean_and_std


Dividing by `n` gives the population standard deviation. Dividing by `n - 1` gives the sample standard deviation, which is a slightly better estimate when your data is a sample from a bigger population.

This difference causes real confusion, because libraries don't agree on the default:

- `np.std(x)` divides by `n` by default (`ddof=0`)
- `pandas.Series.std()` divides by `n - 1` by default (`ddof=1`)

So the same data can give two different numbers depending on which library you use. For large datasets the difference is tiny, but it's worth knowing when your results don't match someone else's.

### Exercise 7: `one_hot`

In [14]:
def one_hot(label, classes):
    if label not in classes:
        raise ValueError(f"unknown label: {label!r}")
    return [1 if c == label else 0 for c in classes]

In [15]:
check_one_hot(one_hot)

PASS  all 4 tests passed for one_hot


Another way is to start with zeros and set one position:

```python
vector = [0] * len(classes)
vector[classes.index(label)] = 1
```

`classes.index` would raise a `ValueError` on its own for a missing label, but the message ("'fish' is not in list") is less helpful than writing your own.

One-hot encoding is how categorical features and class labels get turned into numbers a model can use. Neural network classifiers output one score per class, and the target is compared against a one-hot vector like this.

### Exercise 8: `train_test_split`

In [16]:
def train_test_split(data, test_ratio):
    n_test = round(len(data) * test_ratio)
    split = len(data) - n_test
    return data[:split], data[split:]

In [17]:
check_train_test_split(train_test_split)

PASS  all 5 tests passed for train_test_split


The natural first attempt is `return data[:-n_test], data[-n_test:]`. It works until `n_test` is 0. Then `-0` is just `0`, so `data[:-0]` becomes `data[:0]`, which is empty, and `data[-0:]` becomes the whole list. Your train and test sets swap without any error. Computing the split position from the start of the list avoids this.

The other trap is `int()` instead of `round()`. `100 * 0.29` is `28.999999999999996` in floating point, and `int()` cuts that down to 28.

Slicing always creates new lists, so the original isn't touched.

In practice you shuffle the data before splitting. If your data is sorted by date or by class, taking the last 20% gives you a test set that looks nothing like the training set. The mini project does the shuffle.

### Exercise 9: `confusion_counts`

In [18]:
def confusion_counts(y_true, y_pred):
    counts = {"tp": 0, "fp": 0, "tn": 0, "fn": 0}
    for t, p in zip(y_true, y_pred):
        if t == 1 and p == 1:
            counts["tp"] += 1
        elif t == 0 and p == 1:
            counts["fp"] += 1
        elif t == 0 and p == 0:
            counts["tn"] += 1
        else:
            counts["fn"] += 1
    return counts

In [19]:
check_confusion_counts(confusion_counts)

PASS  all 3 tests passed for confusion_counts


Starting the dictionary with all four keys set to 0 handles the "all keys must be present" requirement and means you never have to check whether a key exists before adding to it.

These four numbers are the foundation of nearly every classification metric: accuracy, precision, recall, F1, specificity and more. scikit-learn's `confusion_matrix` gives you the same numbers arranged in a 2x2 grid.

## Hard

### Exercise 10: `precision_recall_f1`

In [20]:
def precision_recall_f1(y_true, y_pred):
    c = confusion_counts(y_true, y_pred)
    tp, fp, fn = c["tp"], c["fp"], c["fn"]

    precision = tp / (tp + fp) if tp + fp > 0 else 0.0
    recall = tp / (tp + fn) if tp + fn > 0 else 0.0

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1

In [21]:
check_precision_recall_f1(precision_recall_f1)

PASS  all 4 tests passed for precision_recall_f1


Reusing `confusion_counts` keeps this function short and means the counting logic lives in one tested place.

Why do these metrics matter when we have accuracy? Imagine a disease that affects 1 in 100 people. A model that always says "healthy" is 99% accurate and never finds a single sick person. Its recall is 0, which tells you the truth immediately.

Precision and recall usually pull against each other. Lowering the decision threshold catches more positives (higher recall) but also flags more false alarms (lower precision). F1 is their harmonic mean, which is only high when both are high.

scikit-learn handles the divide-by-zero cases with a `zero_division` argument, which defaults to returning 0 with a warning. Same idea as here.

### Exercise 11: `moving_average`

In [22]:
def moving_average(values, window):
    if window < 1 or window > len(values):
        raise ValueError(f"window must be between 1 and {len(values)}, got {window}")
    return [
        sum(values[i:i + window]) / window
        for i in range(len(values) - window + 1)
    ]

In [23]:
check_moving_average(moving_average)

PASS  all 5 tests passed for moving_average


This version is easy to read, but each window sums `window` numbers, so the total work is roughly `len(values) * window`. For a long loss history with a big window, that adds up.

The faster version keeps a running sum. When the window slides one step, add the new value coming in and subtract the old value going out:

In [24]:
def moving_average_fast(values, window):
    if window < 1 or window > len(values):
        raise ValueError(f"window must be between 1 and {len(values)}, got {window}")
    running = sum(values[:window])
    result = [running / window]
    for i in range(window, len(values)):
        running += values[i] - values[i - window]
        result.append(running / window)
    return result

check_moving_average(moving_average_fast)

PASS  all 5 tests passed for moving_average_fast


Now each step does a constant amount of work no matter how big the window is.

One small catch: with floats, adding and subtracting over and over slowly builds up rounding error. For a few thousand values it doesn't matter. For millions, libraries sometimes recompute the sum from scratch every so often to reset the drift.

### Exercise 12: `batch`

In [25]:
def batch(items, batch_size, drop_last=False):
    if batch_size < 1:
        raise ValueError(f"batch_size must be at least 1, got {batch_size}")
    batches = [items[i:i + batch_size] for i in range(0, len(items), batch_size)]
    if drop_last and batches and len(batches[-1]) < batch_size:
        batches.pop()
    return batches

In [26]:
check_batch(batch)

PASS  all 7 tests passed for batch


`range(0, len(items), batch_size)` gives the start index of each batch: 0, 3, 6, and so on. Slicing past the end of a list is safe in Python. It just gives you whatever is left, which is why the last batch comes out shorter without any special handling.

The `batches and ...` check stops `batches[-1]` from crashing on an empty input.

`drop_last` is exactly the name PyTorch's `DataLoader` uses for the same option. People turn it on when a tiny last batch would cause problems, for example with batch normalization, which can't compute meaningful statistics from a batch of one.

In Module 4 we'll rewrite this as a generator, so it can batch data that doesn't fit in memory.

## What's next

The mini project in `04_mini_project.ipynb` builds a k-nearest neighbours classifier using only what's in this module. Several of the functions from these exercises show up there again.